# 🔍 Dataset Explorer & Preparation

> **Mục đích:** Khám phá, kiểm tra, và chuẩn bị dataset trước khi training  
> Chạy notebook này **TRƯỚC** 3 notebook training.

## 📋 Danh sách Dataset Công khai Phù hợp

### 🚗 Vehicle Detection
| Dataset | Ảnh | License | Link |
|---------|-----|---------|------|
| Vehicle Detection (Roboflow) | 14,000+ | CC BY 4.0 | universe.roboflow.com |
| UA-DETRAC | 140,000 | Research | detrac.smileLab.net |
| COCO (car, motorcycle) | 118,000 | CC BY 4.0 | cocodataset.org |
| BDD100K | 100,000 | BSD | bdd-data.berkeley.edu |

### 🪖 Helmet Detection
| Dataset | Ảnh | License | Link |
|---------|-----|---------|------|
| Safety Helmet Detection | 5,000+ | CC BY 4.0 | universe.roboflow.com |
| Hard Hat Workers (Kaggle) | 7,000+ | CC0 | kaggle.com |
| Vietnam Motorcyclist Helmet | 1,500+ | Research | github.com |

### 🔢 License Plate Detection  
| Dataset | Ảnh | License | Link |
|---------|-----|---------|------|
| License Plate Detection (RF) | 6,000+ | CC BY 4.0 | universe.roboflow.com |
| OpenALPR Benchmark | 3,000+ | MIT | github.com/openalpr |
| CCPD (Chinese plates) | 200,000+ | Research | github.com/detectRecog |
| VN License Plate | 1,000+ | Custom | github.com |

## 📦 Cài đặt

In [ ]:
!pip install roboflow ultralytics opencv-python-headless matplotlib --quiet
print('✅ Ready')

## 🔑 Hướng dẫn lấy Roboflow API Key

In [ ]:
print("""
📝 HƯỚNG DẪN LẤY ROBOFLOW API KEY:
====================================
1. Truy cập: https://app.roboflow.com
2. Đăng ký tài khoản miễn phí
3. Vào Settings > API Keys
4. Copy 'Private API Key'
5. Dán vào ô RF_API_KEY ở các notebook training

💡 Free plan: 10,000 images/month download
""")

In [ ]:
# Kiểm tra API key và xem các dataset có thể dùng
RF_API_KEY = 'YOUR_API_KEY'  # <-- THAY VÀO ĐÂY

if RF_API_KEY != 'YOUR_API_KEY':
    from roboflow import Roboflow
    rf = Roboflow(api_key=RF_API_KEY)
    workspace = rf.workspace()
    print(f'✅ Kết nối thành công: {workspace}')
else:
    print('⚠️  Hãy nhập API key trước')

## 🔍 Khám phá Dataset

In [ ]:
# Tải và khám phá dataset mẫu (không cần API key)
# Dùng COCO128 – dataset nhỏ chứa nhiều class xe
import os
os.makedirs('/content/datasets', exist_ok=True)

!wget -q https://ultralytics.com/assets/coco128.zip -O /content/datasets/coco128.zip
!unzip -q /content/datasets/coco128.zip -d /content/datasets/

print('✅ COCO128 đã tải')
!ls /content/datasets/coco128/

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
import numpy as np

# COCO class names
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train',
    'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep',
    'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
    'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard',
    'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork',
    'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair',
    'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv',
    'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave',
    'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase',
    'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

# Classes xe cộ (index trong COCO)
VEHICLE_CLASS_IDS = {1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

def visualize_labels(img_path, label_path, class_names, max_classes=None):
    """Hiển thị ảnh với bounding boxes"""
    img = mpimg.imread(img_path)
    h, w = img.shape[:2]

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.imshow(img)

    COLORS = plt.cm.Set1(np.linspace(0, 1, len(class_names) + 1))

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                if max_classes and cls_id not in max_classes: continue
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                box_w, box_h = bw * w, bh * h

                color = COLORS[cls_id % len(COLORS)]
                rect = patches.Rectangle((x1, y1), box_w, box_h,
                                         linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)

                cls_name = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                ax.text(x1, y1-2, cls_name, color='white', fontsize=9, fontweight='bold',
                       bbox=dict(facecolor=color, alpha=0.7, pad=2))

    ax.axis('off')
    return fig

# Hiển thị ảnh mẫu với labels
images = glob.glob('/content/datasets/coco128/images/train2017/*.jpg')[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img_path in zip(axes.flat, images):
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')
    img = mpimg.imread(img_path)
    h, w = img.shape[:2]

    ax.imshow(img)

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                if cls_id not in [1, 2, 3, 5, 7]: continue  # Chỉ hiện xe
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h

                cls_name = COCO_CLASSES[cls_id]
                rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=2,
                                        edgecolor='lime', facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1-2, cls_name, color='white', fontsize=8,
                       bbox=dict(facecolor='green', alpha=0.7))

    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')

plt.suptitle('COCO128 – Vehicle Classes (bicycle, car, motorcycle, bus, truck)', fontsize=13)
plt.tight_layout()
plt.show()

## 📊 Phân tích phân phối class

In [ ]:
from collections import Counter

def analyze_dataset(dataset_path, class_names):
    """Phân tích số lượng object theo từng class"""
    label_files = glob.glob(f'{dataset_path}/**/*.txt', recursive=True)
    label_files = [f for f in label_files if 'classes' not in f]

    class_counter = Counter()
    total_objects = 0
    total_images = len(label_files)

    for lf in label_files:
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    class_counter[cls_id] += 1
                    total_objects += 1

    print(f'📊 Phân tích Dataset')
    print(f'   Tổng ảnh: {total_images}')
    print(f'   Tổng objects: {total_objects}')
    print(f'   Trung bình obj/ảnh: {total_objects/max(total_images,1):.1f}')
    print()

    # Biểu đồ
    if class_counter:
        labels = [class_names[i] if i < len(class_names) else f'cls_{i}'
                  for i in sorted(class_counter.keys())]
        values = [class_counter[i] for i in sorted(class_counter.keys())]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Bar chart
        colors = plt.cm.Set2(np.linspace(0, 1, len(labels)))
        axes[0].bar(labels, values, color=colors)
        axes[0].set_title('Số lượng object theo class', fontweight='bold')
        axes[0].set_xlabel('Class')
        axes[0].set_ylabel('Số lượng')
        axes[0].tick_params(axis='x', rotation=45)
        for i, v in enumerate(values):
            axes[0].text(i, v + max(values)*0.01, str(v), ha='center', fontsize=9)

        # Pie chart
        axes[1].pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Tỷ lệ phân phối class', fontweight='bold')

        plt.tight_layout()
        plt.show()

    return class_counter

# Phân tích COCO128
stats = analyze_dataset('/content/datasets/coco128', COCO_CLASSES)

## 🔀 Chia train/val nếu cần

In [ ]:
import shutil, random
from pathlib import Path

def split_dataset(source_dir, output_dir, train_ratio=0.8, val_ratio=0.15, test_ratio=0.05, seed=42):
    """
    Chia dataset thành train/val/test
    source_dir: thư mục chứa images/ và labels/
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 0.001, 'Tỷ lệ phải tổng = 1.0'

    random.seed(seed)
    images = glob.glob(f'{source_dir}/images/*')
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    splits = {
        'train': images[:n_train],
        'val':   images[n_train:n_train+n_val],
        'test':  images[n_train+n_val:],
    }

    for split, imgs in splits.items():
        img_out = Path(output_dir) / split / 'images'
        lbl_out = Path(output_dir) / split / 'labels'
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        for img_path in imgs:
            fname = Path(img_path).stem
            ext   = Path(img_path).suffix

            # Copy image
            shutil.copy2(img_path, img_out / (fname + ext))

            # Copy label
            label_src = f'{source_dir}/labels/{fname}.txt'
            if os.path.exists(label_src):
                shutil.copy2(label_src, lbl_out / f'{fname}.txt')

        print(f'  {split}: {len(imgs)} ảnh')

    print(f'\n✅ Dataset đã chia vào: {output_dir}')

# Ví dụ sử dụng:
# split_dataset('/content/my_images', '/content/dataset_split', 0.8, 0.15, 0.05)
print('Hàm split_dataset() đã sẵn sàng')
print('Dùng: split_dataset("path/to/raw", "path/to/output")')

## 📝 Tạo data.yaml tùy chỉnh

In [ ]:
import yaml

def create_yaml(dataset_path, class_names, output_path=None):
    """Tạo file data.yaml cho dataset"""
    config = {
        'path': dataset_path,
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(class_names),
        'names': class_names,
    }

    if output_path is None:
        output_path = f'{dataset_path}/data.yaml'

    with open(output_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

    print(f'✅ Tạo data.yaml tại: {output_path}')
    with open(output_path) as f:
        print(f.read())

    return output_path

# Ví dụ:
# Vehicle YAML
# create_yaml('/content/datasets/vehicle', ['car', 'motorcycle', 'truck', 'bus', 'bicycle'])

# Helmet YAML
# create_yaml('/content/datasets/helmet', ['helmet', 'no_helmet'])

# License Plate YAML
# create_yaml('/content/datasets/license_plate', ['license_plate'])

print('Hàm create_yaml() đã sẵn sàng')

## 🗺️ Roadmap Training

```
Bước 1: Chạy notebook này (00_Dataset_Explorer)
   ↓ Hiểu cấu trúc data, kiểm tra chất lượng
   
Bước 2: Chạy 01_Train_Vehicle_Detection.ipynb
   ↓ Tải dataset từ Roboflow
   ↓ Train YOLOv8s ~50 epochs (~30-45 phút)
   ↓ Lưu vehicle_detection.pt
   
Bước 3: Chạy 02_Train_Helmet_Detection.ipynb  
   ↓ Tải dataset helmet từ Roboflow
   ↓ Train YOLOv8s ~80 epochs (~45-60 phút)
   ↓ Lưu helmet_detection.pt
   
Bước 4: Chạy 03_Train_License_Plate_Detection.ipynb
   ↓ Tải dataset biển số từ Roboflow
   ↓ Train YOLOv8n ~60 epochs (~20-30 phút)
   ↓ Test pipeline YOLOv8 + PaddleOCR
   ↓ Lưu license_plate_detection.pt
   
Bước 5: Download models về máy
   ↓ Google Drive → DATN_TrafficAI/models/
   ↓ Copy vào DATN_VTHUW/models/
   ↓ Restart backend → models tự load
```